# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType 

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_create_date" 
}

# Read data from bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data transformations

Things to fix
1. Trim the strings
2. normalize gndr and martital status fields
3. rename the table and columns with friendly names

## Triming Strings

In [0]:
for field in df.schema.fields: 
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization of Data

In [0]:
df = (
    df
        .withColumn(
            "cst_marital_status",
            F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
             .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
             .otherwise("n/a")
        )
        .withColumn(
            "cst_gndr",
            F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
             .when(F.upper(F.col("cst_gndr")) == "F", "Female")
             .otherwise("n/a")
        )
)

## Renaming Columns

In [0]:
for old_col, new_col in RENAME_MAP.items():
    df = df.withColumnRenamed(old_col, new_col)

# Write it to silver tables

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("silver.crm_cust_info")
)

In [0]:
%sql
SELECT * FROM workspace.silver.crm_cust_info;